In [2]:
import pandas as pd
import numpy as np
import sqlite3
import time
import copy
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader
from scipy.stats import norm
import seaborn as sns
from torch_geometric.data import HeteroData
import torch
import torch_geometric.transforms as T
import matplotlib.pyplot as plt
import sqlite3
import re

ModuleNotFoundError: No module named 'matplotlib'

In [3]:
import sqlite3
import pandas as pd
import numpy as np

In [4]:
conn = sqlite3.connect("logistics_terzo_esperimento.sqlite")
cursor = conn.cursor()

def col_names(table_name):

    cursor.execute(f"PRAGMA table_info({table_name});")
    columns_info = cursor.fetchall()
    column_names = [column[1] for column in columns_info]

    return column_names

In [5]:

try:
    cursor.execute("ALTER TABLE object_Container RENAME COLUMN 'Amount of Handling Units' TO AmountofHandlingUnits;")
    print("Column renamed successfully.")
except sqlite3.OperationalError as e:
    print(f"Error: {e}")
    
try:
    cursor.execute("ALTER TABLE object_TransportDocument RENAME COLUMN 'Amount of Containers' TO AmountofContainers;")
    print("Column renamed successfully.")
except sqlite3.OperationalError as e:
    print(f"Error: {e}")
    
    

Column renamed successfully.
Column renamed successfully.


In [6]:
try:
    cursor.execute("ALTER TABLE object_CustomerOrder RENAME COLUMN 'Amount of Goods' TO AmountofGoods;")
    print("Column renamed successfully.")
except sqlite3.OperationalError as e:
    print(f"Error: {e}")

Column renamed successfully.


In [7]:
table = 'object_CustomerOrder'
cols = col_names(table)
cursor.execute(f'SELECT * FROM {table}')
table = cursor.fetchall()
all_orders = [a[0] for a in table]
#errors = ['co565','co567','co568','co569','co570','co571','co573','co576','co577','co580','co581','co582','co583','co584','co585','co587','co588','co589','co590','co591','co591','co592','co593','co594','co594','co595','co595','co596','co597','co597','co598','co598','co599','co599','co600','co600']
#all_orders = list(set(all_orders) - set(errors))
#all_orders = sorted([int(re.findall(r'\d+', order)[0]) for order in all_orders])
#all_orders = [f'co{order}' for order in all_orders]

In [8]:
order_to_td = {order : 0 for order in all_orders}
order_to_containers = {order : 0 for order in all_orders}
for order in all_orders:
    cursor.execute(f'SELECT ocel_target_id FROM object_object WHERE ocel_source_id = "{order}" AND ocel_qualifier = "TD for CO"')
    td = cursor.fetchall()[0][0]
    order_to_td[order] = td

    cursor.execute(f'SELECT ocel_source_id FROM object_object WHERE ocel_target_id = "{td}" AND ocel_qualifier = "CR for TD"')
    table = cursor.fetchall()
    containers = tuple(a[0] for a in table)
    order_to_containers[order] = containers

In [9]:
def order_to_trace(order, test = False):
    cursor.execute(f'SELECT ocel_target_id FROM object_object WHERE ocel_source_id = "{order}" AND ocel_qualifier = "TD for CO"')
    td = cursor.fetchall()[0][0]

    cursor.execute(f'SELECT ocel_source_id FROM object_object WHERE ocel_target_id = "{td}" AND ocel_qualifier = "CR for TD"')
    table = cursor.fetchall()
    containers = tuple(a[0] for a in table)

    if len(containers) > 1:
        cursor.execute(f'SELECT ocel_target_id FROM object_object WHERE ocel_source_id IN {containers} AND ocel_qualifier = "CR contains HU"')
        table = cursor.fetchall()
        hus = tuple(a[0] for a in table)

        cursor.execute(f'SELECT ocel_source_id FROM object_object WHERE ocel_target_id IN {containers} AND ocel_qualifier = "TR loads CR"')
        table = cursor.fetchall()
        trucks = tuple(set(a[0] for a in table))
    else:
        cursor.execute(f'SELECT ocel_target_id FROM object_object WHERE ocel_source_id = "{containers[0]}" AND ocel_qualifier = "CR contains HU"')
        table = cursor.fetchall()
        hus = tuple(a[0] for a in table)

        cursor.execute(f'SELECT ocel_source_id FROM object_object WHERE ocel_target_id = "{containers[0]}" AND ocel_qualifier = "TR loads CR"')
        table = cursor.fetchall()
        trucks = tuple(set(a[0] for a in table))

    cursor.execute(f'SELECT ocel_target_id FROM object_object WHERE ocel_source_id = "{td}" AND ocel_qualifier LIKE "%VH for TD"')
    table = cursor.fetchall()
    vhs = tuple(a[0] for a in table)

    all_events = hus
    all_events += containers
    all_events += tuple((order,td))

    table = "event_object"
    cols = col_names(table)
    cursor.execute(f"SELECT * FROM {table} WHERE {cols[1]} IN {all_events}")
    table = cursor.fetchall()
    events = tuple([row[0] for row in table])

    table_1 = "event"
    table_2 = "event_map_type"
    cols_1 = col_names(table_1)
    cols_2 = col_names(table_2)
    cursor.execute(f"SELECT {table_1}.{cols_1[0]}, {table_2}.{cols_2[1]} FROM {table_1}\
                    LEFT JOIN {table_2} ON {table_1}.{cols_1[1]} = {table_2}.{cols_2[0]}\
                    WHERE {table_1}.{cols_1[0]} IN {events}")
    tab = cursor.fetchall()

    for idx,row in enumerate(tab):
        temp = "event_" + row[1]
        cursor.execute(f"SELECT * FROM {temp} WHERE {col_names(temp)[0]} = '{row[0]}'")
        table = cursor.fetchall()
        tab[idx] += (table[0][1],)


    tab = sorted(tab, key=lambda x: x[2])
    if test:
        test_idx = a[pd.to_datetime(a[2]) > timestamp].index[0]
        return a[test_idx:]
    return tab,containers,td, vhs

### Graph building

In [10]:
log = []
orders = []
for idx,order in enumerate(all_orders):
    if idx % 200 == 0:
        print(idx)
    temp = order_to_trace(order)[0]
    log.extend(temp)
    orders.extend([idx + 1] * len(temp))
    
log_df = pd.DataFrame(log)
log_df[3] = orders
log_df.to_csv("terzo_esperimento.csv", index = False)

0
200
400
600
800
1000
1200
1400
1600
1800
